# 02 – Data Cleaning and Temporal Alignment

## Objective

This notebook transforms raw observational datasets into a unified, aligned hourly dataset.

Key tasks:

- Convert all temperature units to Celsius
- Standardize timestamp format
- Align time zones
- Handle missing values
- Perform inner and outer joins between stations
- Create a single merged hourly dataframe

The output of this notebook will be stored in `data/processed/`.

This notebook represents the **data engineering layer** of the project.
Without careful alignment, agreement metrics and fusion models will be misleading.

# Notebook 02 — Alignment and Cleaning

In this notebook we prepare our weather datasets so they can be compared and merged.

The raw datasets come from multiple sources:

- **NOAA Daily Summaries** — daily temperature, precipitation, and wind
- **NOAA Hourly Observations** — detailed hourly station measurements
- **METAR Airport Reports** — aviation weather reports from the airport

These datasets use different:

- timestamp formats
- column names
- measurement intervals

Our goal in this notebook is to:

1. Load the raw datasets
2. Inspect their structure
3. Convert timestamps into a consistent format
4. Align datasets so they can be compared

The result will be cleaned datasets saved to:


data/processed/


These cleaned datasets will be used in later notebooks.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path().resolve()

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Raw data folder:", RAW)
print("Processed data folder:", PROCESSED)

Raw data folder: C:\Users\evertj\git\handson-ml3\Weather_Agreement_Lab\data\raw
Processed data folder: C:\Users\evertj\git\handson-ml3\Weather_Agreement_Lab\data\processed


In [3]:
noaa_daily = pd.read_csv(RAW / "noaa_okc_2012_daily.csv")
noaa_hourly = pd.read_csv(RAW / "noaa_okc_2012_hourly.txt")
metar = pd.read_csv(RAW / "metar_okc_2012.csv")

In [4]:
print("NOAA Daily")
display(noaa_daily.head())
print(noaa_daily.columns)

print("\nNOAA Hourly")
display(noaa_hourly.head())
print(noaa_hourly.columns)

print("\nMETAR")
display(metar.head())
print(metar.columns)

NOAA Daily


,STATION,DATE,AWND,PRCP,TMAX,TMIN
0,USW00013967,2012-01-01,5.1,0.0,10.0,1.1
1,USW00013967,2012-01-02,3.3,0.0,7.2,-3.3
2,USW00013967,2012-01-03,5.7,0.0,15.6,-4.4
3,USW00013967,2012-01-04,3.4,0.0,14.4,0.0
4,USW00013967,2012-01-05,4.6,0.0,19.4,-1.7


Index(['STATION', 'DATE', 'AWND', 'PRCP', 'TMAX', 'TMIN'], dtype='str')

NOAA Hourly


,0203723530139672012010100004+35383-097600FM-12+0398KOKC V0203401N012312200019N0160001N1+01671+00061101191ADDGF102991999999999999999999KA1120M+02171KA2180N+00171MD1310721+9999OC101655REMSYN098AAXX 01004 72353 32966 23424 10167 20006 39662 40119 53072 92352 333 10217 20017 91032 555 90100;EQDQ01+096623APOSP
0,0245723530139672012010100527+35389-097601FM-15...
1,0245723530139672012010101527+35389-097601FM-15...
2,0265723530139672012010102527+35389-097601FM-15...
3,0181723530139672012010103004+35383-097600FM-12...
4,0245723530139672012010103527+35389-097601FM-15...


Index(['0203723530139672012010100004+35383-097600FM-12+0398KOKC V0203401N012312200019N0160001N1+01671+00061101191ADDGF102991999999999999999999KA1120M+02171KA2180N+00171MD1310721+9999OC101655REMSYN098AAXX  01004 72353 32966 23424 10167 20006 39662 40119 53072 92352 333 10217 20017 91032 555 90100;EQDQ01+096623APOSP '], dtype='str')

METAR


,station,valid,lon,lat,elevation,tmpf,dwpf,relh,drct,sknt,...,wxcodes,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth
0,OKC,2012-01-01 00:52,-97.6006,35.3889,397.0,59.00,25.00,26.90,330.00,21.00,...,M,M,M,M,29.00,330.00,2012-01-01 00:18,59.00,KOKC 010052Z 33021G27KT 10SM FEW250 15/M04 A29...,M
1,OKC,2012-01-01 01:52,-97.6006,35.3889,397.0,56.00,20.00,24.29,320.00,18.00,...,M,M,M,M,28.00,330.00,2012-01-01 01:47,55.94,KOKC 010152Z 32018G28KT 10SM FEW250 13/M07 A30...,M
2,OKC,2012-01-01 02:52,-97.6006,35.3889,397.0,52.00,28.00,39.40,310.00,28.00,...,M,M,M,M,38.00,310.00,2012-01-01 02:50,51.98,KOKC 010252Z 31028G38KT 8SM CLR 11/M02 A3008 R...,M
3,OKC,2012-01-01 03:52,-97.6006,35.3889,397.0,47.00,31.00,53.47,330.00,32.00,...,M,M,M,M,50.00,340.00,2012-01-01 03:44,36.96,KOKC 010352Z 33032G50KT 8SM CLR 08/M01 A3013 R...,M
4,OKC,2012-01-01 04:52,-97.6006,35.3889,397.0,45.00,31.00,57.63,330.00,24.00,...,M,M,M,M,43.00,330.00,2012-01-01 03:55,35.54,KOKC 010452Z 33024G33KT 10SM CLR 07/M01 A3019 ...,M


Index(['station', 'valid', 'lon', 'lat', 'elevation', 'tmpf', 'dwpf', 'relh',
       'drct', 'sknt', 'p01i', 'alti', 'mslp', 'vsby', 'gust', 'skyc1',
       'skyc2', 'skyc3', 'skyc4', 'skyl1', 'skyl2', 'skyl3', 'skyl4',
       'wxcodes', 'ice_accretion_1hr', 'ice_accretion_3hr',
       'ice_accretion_6hr', 'peak_wind_gust', 'peak_wind_drct',
       'peak_wind_time', 'feel', 'metar', 'snowdepth'],
      dtype='str')


In [9]:
noaa_daily["DATE"] = pd.to_datetime(noaa_daily["DATE"])

In [10]:
metar["valid"] = pd.to_datetime(metar["valid"])

In [13]:
metar = metar.rename(columns={"valid": "DATE"})

In [14]:
metar["DATE"].head()

0   2012-01-01 00:52:00
1   2012-01-01 01:52:00
2   2012-01-01 02:52:00
3   2012-01-01 03:52:00
4   2012-01-01 04:52:00
Name: DATE, dtype: datetime64[us]

In [15]:
noaa_daily["DATE"].head()

0   2012-01-01
1   2012-01-02
2   2012-01-03
3   2012-01-04
4   2012-01-05
Name: DATE, dtype: datetime64[us]

In [16]:
metar["YEAR"] = metar["DATE"].dt.year
metar["MONTH"] = metar["DATE"].dt.month
metar["DAY"] = metar["DATE"].dt.day
metar["HOUR"] = metar["DATE"].dt.hour

In [17]:
# SANITY CHECKS
metar[["DATE","tmpf","relh","sknt"]].head()

,DATE,tmpf,relh,sknt
0,2012-01-01 00:52:00,59.00,26.90,21.00
1,2012-01-01 01:52:00,56.00,24.29,18.00
2,2012-01-01 02:52:00,52.00,39.40,28.00
3,2012-01-01 03:52:00,47.00,53.47,32.00
4,2012-01-01 04:52:00,45.00,57.63,24.00


In [20]:
metar["tmpf"] = pd.to_numeric(metar["tmpf"], errors="coerce")

In [21]:
metar["temp_c"] = (metar["tmpf"] - 32) * 5/9

In [22]:
metar["tmpf"].dtype

dtype('float64')

In [23]:
metar[["tmpf","temp_c"]].head()

,tmpf,temp_c
0,59.0,15.000000
1,56.0,13.333333
2,52.0,11.111111
3,47.0,8.333333
4,45.0,7.222222
